In [1]:
import pandas as pd
import re
import tools

In [2]:
df_kalendarz = pd.read_parquet("dane/interim/kalendarz_pelny_towid.parquet")

### Towary wazone 

In [3]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

wazone_check = df_kalendarz.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count    12481.000000
mean         0.680205
std          6.620668
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         98.976843
Name: PctUlamkowych, dtype: float64


In [4]:


def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [5]:
nazwy_do_sprawdzenia = df_kalendarz[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 195

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 118
       TowId                           NazwaTow  PctUlamkowych
4988   51262                          ANANAS KG       9.090909
5390   52141                 BAKŁAŻAN KG POLSKA       6.666667
3564   25652      BRZUSZKI Z ŁOSOSIA WĘDZONE KG       7.425743
4382   49625                          BURAKI KG       9.330820
9166   77611      CIASTKA MARKIZY KG  DR GERARD       8.152174
...      ...                                ...            ...
2735   16441              SER SALAMI MLEKPOL KG      43.102163
2748   16479   SER Z ORZECHEM WŁOSKIM KG ŁOWICZ      45.038168
11332  80088                         SURÓWKA KG       8.000000
5853   53068  USZKA Z GRZYBAMI KG KUCHNIA POLKI       1.683938
8870   67234               ZIEMNIAKI SŁODKIE KG      35.164835

[118 rows x 3 columns]


In [6]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    8463825
True      338690
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
PIECZYWO                          3
WARZYWA KISZONE                   3
RYBY WĘDZONE WAGA /ryby           3
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
MIĘSO, WĘDLINY I GARMAŻERKA       1
Name: count, dtype: int64


In [7]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    8463825
True      338690
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
PIECZYWO                          3
WARZYWA KISZONE                   3
RYBY WĘDZONE WAGA /ryby           3
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
MIĘSO, WĘDLINY I GARMAŻERKA       1
Name: count, dtype: int64


In [8]:
df_kalendarz.to_parquet(
    "dane/interim/fact_inka_hard_flagged_wazone.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_kalendarz.shape}")

Zapisano: (8802515, 34)


In [ ]:
# suma kontrolna
nazwa_pliku = "fact_inka_hard_flagged_wazone.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
#Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   7c2653bba73d224182b9d5cffa792d5827a17f0e1440a2e3c27214eb3ad0191a

Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   7c2653bba73d224182b9d5cffa792d5827a17f0e1440a2e3c27214eb3ad0191a
